<a href="https://colab.research.google.com/github/VISHWAJA-028/ML_Multimodality/blob/main/pcos_predictor_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import importlib
import io
import os
import pickle
import platform
import re
import tempfile
import warnings

import numpy as np
import pandas as pd


BUNDLE_FORMAT = 1

DISCLAIMER = (
    "Screening aid only - not a medical diagnosis. "
    "Please confirm with a qualified clinician."
)

In [4]:

# ============================================================
# KEY FIELDS
# ============================================================

KEY_FIELDS = [
    "hair_growth_Y_N",
    "Weight_gain_Y_N",
    "Skin_darkening_Y_N",
    "Follicle_No_R",
    "Follicle_No_L",
    "AMH_ng_mL"
]

_DERIVED = {
    "Follicle_Total",
    "Avg_F_Size_Mean"
}

_DUMMY_PREFIXES = (
    "Blood_Group_",
    "Cycle_R_I_",
    "BMI_Category_"
)

_IGNORED_INPUTS = {
    "Sl_No",
    "Patient_File_No",
    "Source_Dataset",
    "PCOS_Y_N"
}

In [6]:




# ============================================================
# BLOOD GROUP / CYCLE MAPPINGS
# ============================================================

_BG_COL = {
    "A+": "Blood_Group_A",
    "A-": "Blood_Group_A_1",
    "AB+": "Blood_Group_AB",
    "AB-": "Blood_Group_AB_1",
    "B+": "Blood_Group_B",
    "B-": "Blood_Group_B_1",
    "O+": "Blood_Group_O",
    "O-": "Blood_Group_O_1"
}

_BG_CODES = {
    "11": "A+",
    "12": "A-",
    "13": "B+",
    "14": "B-",
    "15": "O+",
    "16": "O-",
    "17": "AB+",
    "18": "AB-"
}

_CYCLE_CODES = {
    "2": "R",
    "4": "I",
    "R": "R",
    "I": "I",
    "REGULAR": "R",
    "IRREGULAR": "I"
}


# ============================================================
# 1. FEATURE ENGINEERING
# ============================================================

def clean_name(col):
    """
    Convert original column names into cleaned names.

    Example:
    Follicle No. (L) -> Follicle_No_L
    """

    c = re.sub(
        r"[^A-Za-z0-9_]",
        "_",
        str(col)
    )

    c = re.sub(
        r"_{2,}",
        "_",
        c
    )

    return c.strip("_")


def _as_code(v):
    """
    Convert values such as:
    12
    12.0
    '12'

    into:
    '12'
    """

    if v is None:
        return None

    if isinstance(v, float) and np.isnan(v):
        return None

    s = str(v).strip()

    try:
        f = float(s)

        if f == int(f):
            return str(int(f))

    except ValueError:
        pass

    return s.upper()


def _yn(v):

    if v is None:
        return np.nan

    if v is pd.NA:
        return np.nan

    if isinstance(v, float) and np.isnan(v):
        return np.nan

    return {
        "Y": 1,
        "YES": 1,
        "1": 1,
        "1.0": 1,
        "TRUE": 1,

        "N": 0,
        "NO": 0,
        "0": 0,
        "0.0": 0,
        "FALSE": 0
    }.get(
        str(v).strip().upper(),
        np.nan
    )


def _norm_blood_group(v):

    s = _as_code(v)

    if s is None:
        return None

    s = _BG_CODES.get(
        s,
        s
    )

    if s in _BG_COL:
        return s

    return None


def _norm_cycle(v):

    s = _as_code(v)

    if s is None:
        return None

    return _CYCLE_CODES.get(s)


def _fix_double_decimal(x):

    s = str(x)

    if s.count(".") > 1:

        i = s.index(".")

        s = (
            s[:i + 1]
            + s[i + 1:].replace(".", "")
        )

    return s


def _bmi_category(b):

    if pd.isna(b):
        return None

    if b < 18.5:
        return "Underweight"

    if b < 25:
        return "Normal"

    if b < 30:
        return "Overweight"

    return "Obese"


def engineer_features(raw, columns):
    """
    Convert user input into exactly the same
    feature structure used during model training.
    """

    if isinstance(
        raw,
        pd.DataFrame
    ):

        df = raw.copy()

    elif isinstance(
        raw,
        pd.Series
    ):

        df = raw.to_frame().T

    elif isinstance(
        raw,
        dict
    ):

        df = pd.DataFrame([raw])

    else:

        raise TypeError(
            "features must be a dict, "
            "pandas Series or DataFrame"
        )

    df = df.reset_index(
        drop=True
    )

    df.columns = [
        clean_name(c)
        for c in df.columns
    ]

    df = df.loc[
        :,
        ~df.columns.duplicated()
    ]

    blank = pd.Series(
        np.nan,
        index=df.index,
        dtype=object
    )

    dummies = [
        c
        for c in columns
        if c.startswith(
            _DUMMY_PREFIXES
        )
    ]

    yn_cols = [
        c
        for c in columns
        if c.endswith("_Y_N")
    ]

    numeric = [
        c
        for c in columns
        if c not in dummies
        and c not in yn_cols
        and c not in _DERIVED
    ]

    out = pd.DataFrame(
        index=df.index
    )

    # --------------------------------------------------------
    # Numeric columns
    # --------------------------------------------------------

    for c in numeric:

        s = (
            df[c]
            if c in df.columns
            else blank
        )

        if c == "II_beta_HCG_mIU_mL":

            s = s.map(
                _fix_double_decimal
            )

        out[c] = pd.to_numeric(
            s,
            errors="coerce"
        ).astype(float)

    # --------------------------------------------------------
    # Derived features
    # --------------------------------------------------------

    if {
        "BMI",
        "Weight_Kg",
        "Height_Cm"
    } <= set(out.columns):

        out["BMI"] = out["BMI"].fillna(
            out["Weight_Kg"]
            / (out["Height_Cm"] / 100) ** 2
        )

    if {
        "Waist_Hip_Ratio",
        "Waist_inch",
        "Hip_inch"
    } <= set(out.columns):

        out["Waist_Hip_Ratio"] = (
            out["Waist_Hip_Ratio"].fillna(
                out["Waist_inch"]
                / out["Hip_inch"].replace(
                    0,
                    np.nan
                )
            )
        )

    if {
        "FSH_LH",
        "FSH_mIU_mL",
        "LH_mIU_mL"
    } <= set(out.columns):

        out["FSH_LH"] = (
            out["FSH_LH"].fillna(
                out["FSH_mIU_mL"]
                / out["LH_mIU_mL"].replace(
                    0,
                    np.nan
                )
            )
        )

    # --------------------------------------------------------
    # Follicle features
    # --------------------------------------------------------

    if (
        "Follicle_Total" in columns
        and {
            "Follicle_No_L",
            "Follicle_No_R"
        } <= set(out.columns)
    ):

        out["Follicle_Total"] = (
            out["Follicle_No_L"]
            + out["Follicle_No_R"]
        )

    if (
        "Avg_F_Size_Mean" in columns
        and {
            "Avg_F_size_L_mm",
            "Avg_F_size_R_mm"
        } <= set(out.columns)
    ):

        out["Avg_F_Size_Mean"] = (
            out[
                [
                    "Avg_F_size_L_mm",
                    "Avg_F_size_R_mm"
                ]
            ].mean(axis=1)
        )

    # --------------------------------------------------------
    # Yes / No columns
    # --------------------------------------------------------

    for c in yn_cols:

        out[c] = (
            df[c]
            if c in df.columns
            else blank
        ).map(
            _yn
        ).astype(float)

    # --------------------------------------------------------
    # Blood group / cycle / BMI
    # --------------------------------------------------------

    bg = (
        df["Blood_Group"]
        if "Blood_Group" in df.columns
        else blank
    ).map(
        _norm_blood_group
    )

    cy = (
        df["Cycle_R_I"]
        if "Cycle_R_I" in df.columns
        else blank
    ).map(
        _norm_cycle
    )

    bc = (
        out["BMI"].map(
            _bmi_category
        )
        if "BMI" in out.columns
        else blank
    )

    # --------------------------------------------------------
    # Blood group dummy variables
    # --------------------------------------------------------

    for label, col in _BG_COL.items():

        out[col] = (
            bg == label
        ).astype(float)

    out["Blood_Group_nan"] = (
        bg.isna()
    ).astype(float)

    # --------------------------------------------------------
    # Cycle dummy variables
    # --------------------------------------------------------

    out["Cycle_R_I_I"] = (
        cy == "I"
    ).astype(float)

    out["Cycle_R_I_R"] = (
        cy == "R"
    ).astype(float)

    out["Cycle_R_I_nan"] = (
        cy.isna()
    ).astype(float)

    # --------------------------------------------------------
    # BMI dummy variables
    # --------------------------------------------------------

    for cat in (
        "Normal",
        "Obese",
        "Overweight",
        "Underweight"
    ):

        out[
            f"BMI_Category_{cat}"
        ] = (
            bc == cat
        ).astype(float)

    out["BMI_Category_nan"] = (
        bc.isna()
    ).astype(float)

    # --------------------------------------------------------
    # Match training columns exactly
    # --------------------------------------------------------

    out = out.reindex(
        columns=list(columns)
    )

    fields = (
        numeric
        + yn_cols
        + [
            "Blood_Group",
            "Cycle_R_I"
        ]
    )

    info = {
        "fields": fields,
        "missing": None,
        "unknown": None
    }

    # --------------------------------------------------------
    # Missing / unknown fields
    # --------------------------------------------------------

    if len(df) == 1:

        have = {
            c: bool(
                pd.notna(
                    out.at[0, c]
                )
            )
            for c in numeric + yn_cols
        }

        have["Blood_Group"] = bool(
            pd.notna(
                bg.iloc[0]
            )
        )

        have["Cycle_R_I"] = bool(
            pd.notna(
                cy.iloc[0]
            )
        )

        info["missing"] = [
            c
            for c in fields
            if not have[c]
        ]

        known = (
            set(columns)
            | {
                "Blood_Group",
                "Cycle_R_I"
            }
            | _IGNORED_INPUTS
        )

        info["unknown"] = [
            c
            for c in df.columns
            if c not in known
        ]

    return out, info


# ============================================================
# 2. THRESHOLD HELPER
# ============================================================

def tune_threshold(
    y_true_pcos,
    p_pcos,
    min_recall=0.95
):

    from sklearn.metrics import roc_curve

    fpr, tpr, thr = roc_curve(
        y_true_pcos,
        p_pcos
    )

    ok = np.where(
        tpr >= min_recall
    )[0]

    if len(ok) == 0:

        raise ValueError(
            "min_recall not reachable"
        )

    return float(
        thr[
            ok[
                np.argmin(
                    fpr[ok]
                )
            ]
        ]
    )


# ============================================================
# 3. VERSION INFORMATION
# ============================================================

def _versions():

    v = {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__
    }

    for m in (
        "sklearn",
        "lightgbm",
        "xgboost",
        "tensorflow"
    ):

        try:

            v[m] = (
                importlib
                .import_module(m)
                .__version__
            )

        except Exception:

            pass

    return v


# ============================================================
# 4. CREATE BUNDLE
# ============================================================

def create_bundle(
    path,
    tab_model,
    tab_imputer,
    tab_scaler,
    tab_columns,
    keras_model_path,
    img_size=(224, 224),
    image_p_is_pcos=False,
    image_threshold=0.5,
    image_tta=1,
    tab_threshold=0.5,
    meta=None
):

    with open(
        keras_model_path,
        "rb"
    ) as f:

        keras_bytes = f.read()

    bundle = {

        "format": BUNDLE_FORMAT,

        "tabular": {

            "model": tab_model,

            "imputer": tab_imputer,

            "scaler": tab_scaler,

            "columns": list(
                tab_columns
            ),

            "threshold": float(
                tab_threshold
            )
        },

        "image": {

            "keras_bytes": keras_bytes,

            "img_size": tuple(
                img_size
            ),

            # Current image model:
            # infected = 0
            # noninfected = 1
            #
            # Therefore CNN output is
            # P(noninfected)

            "p_is_pcos": bool(
                image_p_is_pcos
            ),

            "threshold": float(
                image_threshold
            ),

            "tta": int(
                image_tta
            )
        },

        "meta": {
            "versions": _versions(),
            **(meta or {})
        }
    }

    with open(
        path,
        "wb"
    ) as f:

        pickle.dump(
            bundle,
            f,
            protocol=pickle.HIGHEST_PROTOCOL
        )

    return path


# ============================================================
# 5. PCOS PREDICTOR
# ============================================================

class PCOSPredictor:

    def __init__(
        self,
        bundle
    ):

        if bundle.get(
            "format"
        ) != BUNDLE_FORMAT:

            raise ValueError(
                "Unsupported bundle format"
            )

        self.tab = bundle[
            "tabular"
        ]

        self.img = bundle[
            "image"
        ]

        self.meta = bundle.get(
            "meta",
            {}
        )

        self._img_model = None


    # ========================================================
    # LOAD BUNDLE
    # ========================================================

    @classmethod
    def load(
        cls,
        path
    ):

        with open(
            path,
            "rb"
        ) as f:

            bundle = pickle.load(f)

        saved = (
            bundle
            .get("meta", {})
            .get("versions", {})
        )

        now = _versions()

        for lib in (
            "sklearn",
            "lightgbm",
            "tensorflow"
        ):

            if (
                lib in saved
                and lib in now
                and saved[lib] != now[lib]
            ):

                warnings.warn(
                    f"{lib} version differs "
                    f"(saved {saved[lib]}, "
                    f"now {now[lib]}); "
                    "results may change or "
                    "loading may fail."
                )

        return cls(
            bundle
        )


    # ========================================================
    # STRUCTURED MODEL
    # ========================================================

    def _tab_probs(
        self,
        X
    ):

        cols = self.tab[
            "columns"
        ]

        Xi = pd.DataFrame(
            self.tab["imputer"].transform(X),
            columns=cols,
            index=X.index
        )

        Xs = pd.DataFrame(
            self.tab["scaler"].transform(Xi),
            columns=cols,
            index=X.index
        )

        return self.tab[
            "model"
        ].predict_proba(
            Xs
        )[:, 1]


    def predict_structured(
        self,
        features
    ):

        X, info = engineer_features(
            features,
            self.tab["columns"]
        )

        if len(X) != 1:

            raise ValueError(
                "Use predict_structured_batch() "
                "for several rows"
            )

        p = float(
            self._tab_probs(X)[0]
        )

        thr = self.tab[
            "threshold"
        ]

        missing = info[
            "missing"
        ]

        completeness = (
            1
            - len(missing)
            / len(info["fields"])
        )

        key_missing = [
            c
            for c in KEY_FIELDS
            if c in missing
        ]

        warns = []

        if completeness < 0.6:

            warns.append(
                "Many fields are missing; "
                "they were filled with training "
                "medians, so treat this as "
                "low-confidence."
            )

        if key_missing:

            warns.append(
                "Strongest predictors not provided: "
                + ", ".join(key_missing)
            )

        if info["unknown"]:

            warns.append(
                "Ignored unrecognised fields "
                "(typo?): "
                + ", ".join(
                    info["unknown"]
                )
            )

        return {

            "label":
                "PCOS"
                if p >= thr
                else "Non-PCOS",

            "prob_pcos":
                round(p, 4),

            "threshold":
                thr,

            "input_completeness":
                round(
                    completeness,
                    2
                ),

            "missing_fields":
                missing,

            "warnings":
                warns
        }


    def predict_structured_batch(
        self,
        df
    ):

        X, _ = engineer_features(
            df,
            self.tab["columns"]
        )

        p = self._tab_probs(
            X
        )

        return pd.DataFrame({

            "prob_pcos":
                p.round(4),

            "label":
                np.where(
                    p >= self.tab["threshold"],
                    "PCOS",
                    "Non-PCOS"
                )
        })


    # ========================================================
    # IMAGE MODEL
    # ========================================================

    def _get_image_model(
        self
    ):

        if self._img_model is None:

            from tensorflow import keras

            with tempfile.NamedTemporaryFile(
                suffix=".keras",
                delete=False
            ) as f:

                f.write(
                    self.img[
                        "keras_bytes"
                    ]
                )

                tmp = f.name

            try:

                self._img_model = (
                    keras.models.load_model(
                        tmp,
                        compile=False
                    )
                )

            finally:

                os.remove(
                    tmp
                )

        return self._img_model


    def _load_image(
        self,
        image
    ):

        from PIL import Image

        if isinstance(
            image,
            (str, os.PathLike)
        ):

            im = Image.open(
                image
            )

        elif isinstance(
            image,
            (bytes, bytearray)
        ):

            im = Image.open(
                io.BytesIO(
                    image
                )
            )

        elif isinstance(
            image,
            Image.Image
        ):

            im = image

        elif isinstance(
            image,
            np.ndarray
        ):

            im = Image.fromarray(
                image.astype(
                    "uint8"
                )
            )

        else:

            raise TypeError(
                "image must be a path, "
                "bytes, PIL image or numpy array"
            )

        arr = np.asarray(
            im.convert("RGB")
        )

        size = tuple(
            self.img[
                "img_size"
            ]
        )

        try:

            import tensorflow as tf

            arr = (
                tf.image
                .resize(
                    arr,
                    size
                )
                .numpy()
            )

        except ImportError:

            arr = np.asarray(
                Image.fromarray(
                    arr
                ).resize(
                    size[::-1],
                    Image.BILINEAR
                )
            )

        return arr.astype(
            "float32"
        )


    def predict_image(
        self,
        image
    ):

        arr = self._load_image(
            image
        )

        n = max(
            1,
            int(
                self.img[
                    "tta"
                ]
            )
        )

        raw = float(
            self._get_image_model()
            .predict(
                np.repeat(
                    arr[None],
                    n,
                    axis=0
                ),
                verbose=0
            )
            .ravel()
            .mean()
        )

        # Current model output:
        # P(noninfected)
        #
        # Convert to P(PCOS)

        if self.img[
            "p_is_pcos"
        ]:

            p = raw

        else:

            p = 1.0 - raw

        thr = self.img[
            "threshold"
        ]

        is_pcos = (
            p >= thr
        )

        return {

            "label":
                "PCOS"
                if is_pcos
                else "Non-PCOS",

            "prob_pcos":
                round(
                    p,
                    4
                ),

            "threshold":
                thr,

            "passes_averaged":
                n
        }


    # ========================================================
    # COMBINED PREDICTION
    # ========================================================

    def predict(
        self,
        features=None,
        image=None,
        tab_weight=0.5
    ):

        if (
            features is None
            and image is None
        ):

            raise ValueError(
                "Give features=..., "
                "image=..., or both"
            )

        out = {
            "disclaimer":
                DISCLAIMER
        }

        t = (
            self.predict_structured(
                features
            )
            if features is not None
            else None
        )

        i = (
            self.predict_image(
                image
            )
            if image is not None
            else None
        )

        if t:
            out["structured"] = t

        if i:
            out["image"] = i

        if t and i:

            out["mode"] = "both"

        elif t:

            out["mode"] = "structured"

        else:

            out["mode"] = "image"

        # ----------------------------------------------------
        # Combine both models
        # ----------------------------------------------------

        if t and i:

            p = (
                tab_weight
                * t["prob_pcos"]
                +
                (1 - tab_weight)
                * i["prob_pcos"]
            )

            out["combined"] = {

                "label":
                    "PCOS"
                    if p >= 0.5
                    else "Non-PCOS",

                "prob_pcos":
                    round(
                        p,
                        4
                    ),

                "agree":
                    t["label"]
                    == i["label"],

                "note":
                    (
                        "Heuristic average: the two "
                        "models were trained on different "
                        "patients, so the weights are not "
                        "learned. When they disagree, "
                        "refer for clinical review."
                    )
            }

        return out


    # ========================================================
    # AUTO ROUTING
    # ========================================================

    def predict_any(
        self,
        x
    ):

        """
        Automatically route the input:

        dict / Series / DataFrame
            -> structured model

        anything else
            -> image model
        """

        if isinstance(
            x,
            (
                dict,
                pd.Series,
                pd.DataFrame
            )
        ):

            return self.predict(
                features=x
            )

        return self.predict(
            image=x
        )


    # ========================================================
    # DESCRIBE
    # ========================================================

    def describe(
        self
    ):

        print(
            "Structured columns:",
            len(
                self.tab[
                    "columns"
                ]
            ),
            "| threshold",
            self.tab[
                "threshold"
            ]
        )

        print(
            "Image size:",
            self.img[
                "img_size"
            ],

            "| P is PCOS:",
            self.img[
                "p_is_pcos"
            ],

            "| threshold:",
            self.img[
                "threshold"
            ],

            "| TTA:",
            self.img[
                "tta"
            ]
        )

        print(
            "Meta:",
            {
                k: v
                for k, v in self.meta.items()
                if k != "versions"
            }
        )

        print(
            "Saved with:",
            self.meta.get(
                "versions"
            )
        )

In [8]:
bundle_path = "/content/drive/MyDrive/PCOS/artifacts/pcos_bundle.pkl"

predictor = PCOSPredictor.load(bundle_path)

print("✅ Multimodal PCOS predictor loaded!")
predictor.describe()

✅ Multimodal PCOS predictor loaded!
Structured columns: 58 | threshold 0.5
Image size: (224, 224) | P is PCOS: False | threshold: 0.5 | TTA: 1
Meta: {'image_backbone': 'ResNet50', 'image_test_metrics': {'accuracy': 0.8721719457013575, 'precision': 0.7684426229508197, 'recall': 1.0, 'f1_score': 0.8690614136732329, 'roc_auc': 0.9989829731499673}}
Saved with: {'python': '3.13.15', 'numpy': '2.1.3', 'pandas': '2.2.3', 'sklearn': '1.6.1', 'lightgbm': '4.6.0', 'xgboost': '3.4.1', 'tensorflow': '2.20.0'}


In [12]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/PCOS/archive (16).zip"
extract_path = "/content/dataset"

print("ZIP exists:", os.path.exists(zip_path))

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Dataset extracted successfully!")

ZIP exists: True
✅ Dataset extracted successfully!


In [17]:
import os


print(os.listdir("/content/dataset"))
path = "/content/dataset/archive (7)"

print(os.listdir(path))
pcos_path = "/content/dataset/archive (7)/PCOS"

print(os.listdir(pcos_path))

['archive (7)']
['PCOS']
['noninfected', 'infected']


In [18]:
import glob

infected_images = glob.glob(
    "/content/dataset/archive (7)/PCOS/infected/*"
)

print("Number of infected images:", len(infected_images))
print("First image:", infected_images[0])

Number of infected images: 6784
First image: /content/dataset/archive (7)/PCOS/infected/image11359.jpg


In [19]:
image_path = infected_images[0]

result = predictor.predict_image(image_path)

print(result)

{'label': 'Non-PCOS', 'prob_pcos': 0.1386, 'threshold': 0.5, 'passes_averaged': 1}


In [20]:
patient_data = {
    "Age (yrs)": 27,
    "Weight (Kg)": 68,
    "Height(Cm)": 155,
    "Cycle(R/I)": "I",
    "hair growth(Y/N)": "Y",
    "Weight gain(Y/N)": "Y",
    "Skin darkening (Y/N)": "N",
    "Follicle No. (L)": 14,
    "Follicle No. (R)": 16,
    "AMH(ng/mL)": 6.5
}

structured_result = predictor.predict_structured(patient_data)

print(structured_result)

{'label': 'PCOS', 'prob_pcos': 0.9998, 'threshold': 0.5, 'input_completeness': 0.27, 'missing_fields': ['Pulse_rate_bpm', 'RR_breaths_min', 'Hb_g_dl', 'Cycle_length_days', 'Marraige_Status_Yrs', 'No_of_aborptions', 'I_beta_HCG_mIU_mL', 'II_beta_HCG_mIU_mL', 'FSH_mIU_mL', 'LH_mIU_mL', 'FSH_LH', 'Hip_inch', 'Waist_inch', 'Waist_Hip_Ratio', 'TSH_mIU_L', 'PRL_ng_mL', 'Vit_D3_ng_mL', 'PRG_ng_mL', 'RBS_mg_dl', 'BP_Systolic_mmHg', 'BP_Diastolic_mmHg', 'Avg_F_size_L_mm', 'Avg_F_size_R_mm', 'Endometrium_mm', 'Pregnant_Y_N', 'Hair_loss_Y_N', 'Pimples_Y_N', 'Fast_food_Y_N', 'Reg_Exercise_Y_N', 'Blood_Group'], 'warnings': ['Many fields are missing; they were filled with training medians, so treat this as low-confidence.']}


In [21]:
image_path = infected_images[0]

final_result = predictor.predict(
    features=patient_data,
    image=image_path
)

print("===== FINAL MULTIMODAL RESULT =====")
print(final_result)

===== FINAL MULTIMODAL RESULT =====
{'disclaimer': 'Screening aid only - not a medical diagnosis. Please confirm with a qualified clinician.', 'structured': {'label': 'PCOS', 'prob_pcos': 0.9998, 'threshold': 0.5, 'input_completeness': 0.27, 'missing_fields': ['Pulse_rate_bpm', 'RR_breaths_min', 'Hb_g_dl', 'Cycle_length_days', 'Marraige_Status_Yrs', 'No_of_aborptions', 'I_beta_HCG_mIU_mL', 'II_beta_HCG_mIU_mL', 'FSH_mIU_mL', 'LH_mIU_mL', 'FSH_LH', 'Hip_inch', 'Waist_inch', 'Waist_Hip_Ratio', 'TSH_mIU_L', 'PRL_ng_mL', 'Vit_D3_ng_mL', 'PRG_ng_mL', 'RBS_mg_dl', 'BP_Systolic_mmHg', 'BP_Diastolic_mmHg', 'Avg_F_size_L_mm', 'Avg_F_size_R_mm', 'Endometrium_mm', 'Pregnant_Y_N', 'Hair_loss_Y_N', 'Pimples_Y_N', 'Fast_food_Y_N', 'Reg_Exercise_Y_N', 'Blood_Group'], 'warnings': ['Many fields are missing; they were filled with training medians, so treat this as low-confidence.']}, 'image': {'label': 'Non-PCOS', 'prob_pcos': 0.1386, 'threshold': 0.5, 'passes_averaged': 1}, 'mode': 'both', 'combined': 

In [22]:
print("===== FINAL RESULT =====")
print(final_result)

print("\n===== PREDICTION =====")
print("Label:", final_result.get("label"))
print("PCOS Probability:", final_result.get("prob_pcos"))

===== FINAL RESULT =====
{'disclaimer': 'Screening aid only - not a medical diagnosis. Please confirm with a qualified clinician.', 'structured': {'label': 'PCOS', 'prob_pcos': 0.9998, 'threshold': 0.5, 'input_completeness': 0.27, 'missing_fields': ['Pulse_rate_bpm', 'RR_breaths_min', 'Hb_g_dl', 'Cycle_length_days', 'Marraige_Status_Yrs', 'No_of_aborptions', 'I_beta_HCG_mIU_mL', 'II_beta_HCG_mIU_mL', 'FSH_mIU_mL', 'LH_mIU_mL', 'FSH_LH', 'Hip_inch', 'Waist_inch', 'Waist_Hip_Ratio', 'TSH_mIU_L', 'PRL_ng_mL', 'Vit_D3_ng_mL', 'PRG_ng_mL', 'RBS_mg_dl', 'BP_Systolic_mmHg', 'BP_Diastolic_mmHg', 'Avg_F_size_L_mm', 'Avg_F_size_R_mm', 'Endometrium_mm', 'Pregnant_Y_N', 'Hair_loss_Y_N', 'Pimples_Y_N', 'Fast_food_Y_N', 'Reg_Exercise_Y_N', 'Blood_Group'], 'warnings': ['Many fields are missing; they were filled with training medians, so treat this as low-confidence.']}, 'image': {'label': 'Non-PCOS', 'prob_pcos': 0.1386, 'threshold': 0.5, 'passes_averaged': 1}, 'mode': 'both', 'combined': {'label': '

In [23]:
print("Final Label:", final_result["combined"]["label"])
print("Final PCOS Probability:", final_result["combined"]["prob_pcos"])

Final Label: PCOS
Final PCOS Probability: 0.5692


In [24]:
import pandas as pd

excel_path = "/content/drive/MyDrive/PCOS/pocs_final.xlsx"

df = pd.read_excel(excel_path)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head(2))

Rows: 3556
Columns: 45


,Sl. No,Patient File No.,PCOS (Y/N),Age (yrs),Weight (Kg),Height(Cm),BMI,Blood Group,Pulse rate(bpm),RR (breaths/min),...,Fast food (Y/N),Reg.Exercise(Y/N),BP _Systolic (mmHg),BP _Diastolic (mmHg),Follicle No. (L),Follicle No. (R),Avg. F size (L) (mm),Avg. F size (R) (mm),Endometrium (mm),Source_Dataset
0,1,P-10000,0,30,69.979147,167.708055,23.185569,12,72,22,...,0.0,1,105.483401,76.096379,2,4,10.0,13.0,6.176029,NaN
1,2,P-10001,0,36,63.711688,154.055877,25.441392,13,70,18,...,0.0,1,115.883740,79.117243,2,3,13.0,11.0,6.824718,NaN


In [25]:
print(df.columns.tolist())

['Sl. No', 'Patient File No.', 'PCOS (Y/N)', 'Age (yrs)', 'Weight (Kg)', 'Height(Cm)', 'BMI', 'Blood Group', 'Pulse rate(bpm)', 'RR (breaths/min)', 'Hb(g/dl)', 'Cycle(R/I)', 'Cycle length(days)', 'Marraige Status (Yrs)', 'Pregnant(Y/N)', 'No. of aborptions', 'I beta-HCG(mIU/mL)', 'II beta-HCG(mIU/mL)', 'FSH(mIU/mL)', 'LH(mIU/mL)', 'FSH/LH', 'Hip(inch)', 'Waist(inch)', 'Waist:Hip Ratio', 'TSH (mIU/L)', 'AMH(ng/mL)', 'PRL(ng/mL)', 'Vit D3 (ng/mL)', 'PRG(ng/mL)', 'RBS(mg/dl)', 'Weight gain(Y/N)', 'hair growth(Y/N)', 'Skin darkening (Y/N)', 'Hair loss(Y/N)', 'Pimples(Y/N)', 'Fast food (Y/N)', 'Reg.Exercise(Y/N)', 'BP _Systolic (mmHg)', 'BP _Diastolic (mmHg)', 'Follicle No. (L)', 'Follicle No. (R)', 'Avg. F size (L) (mm)', 'Avg. F size (R) (mm)', 'Endometrium (mm)', 'Source_Dataset']


In [26]:
# Select the first patient
patient_row = df.iloc[0]

print(patient_row)

Sl. No                            1
Patient File No.            P-10000
PCOS (Y/N)                        0
Age (yrs)                        30
Weight (Kg)               69.979147
Height(Cm)               167.708055
BMI                       23.185569
Blood Group                      12
Pulse rate(bpm)                  72
RR (breaths/min)                 22
Hb(g/dl)                       12.0
Cycle(R/I)                        4
Cycle length(days)                5
Marraige Status (Yrs)          10.0
Pregnant(Y/N)                     1
No. of aborptions                 1
I beta-HCG(mIU/mL)            455.8
II beta-HCG(mIU/mL)           121.8
FSH(mIU/mL)                     5.7
LH(mIU/mL)                     2.37
FSH/LH                         2.41
Hip(inch)                      37.0
Waist(inch)                    32.0
Waist:Hip Ratio                0.86
TSH (mIU/L)                    2.36
AMH(ng/mL)                     5.76
PRL(ng/mL)                    10.79
Vit D3 (ng/mL)              

In [27]:
patient_data = patient_row.drop(
    ["Sl. No", "Patient File No.", "PCOS (Y/N)", "Source_Dataset"]
).to_dict()

print("Number of input fields:", len(patient_data))
print(patient_data)

Number of input fields: 41
{'Age (yrs)': 30, 'Weight (Kg)': 69.97914713032205, 'Height(Cm)': 167.7080554233576, 'BMI': 23.18556936301909, 'Blood Group': '12', 'Pulse rate(bpm)': 72, 'RR (breaths/min)': 22, 'Hb(g/dl)': 12.0, 'Cycle(R/I)': '4', 'Cycle length(days)': 5, 'Marraige Status (Yrs)': 10.0, 'Pregnant(Y/N)': '1', 'No. of aborptions': 1, 'I beta-HCG(mIU/mL)': 455.8, 'II beta-HCG(mIU/mL)': '121.8', 'FSH(mIU/mL)': 5.7, 'LH(mIU/mL)': 2.37, 'FSH/LH': 2.41, 'Hip(inch)': 37.0, 'Waist(inch)': 32.0, 'Waist:Hip Ratio': 0.86, 'TSH (mIU/L)': 2.36, 'AMH(ng/mL)': '5.76', 'PRL(ng/mL)': 10.79, 'Vit D3 (ng/mL)': 28.7, 'PRG(ng/mL)': 0.25, 'RBS(mg/dl)': 108.0, 'Weight gain(Y/N)': '0', 'hair growth(Y/N)': '1', 'Skin darkening (Y/N)': '0', 'Hair loss(Y/N)': '0', 'Pimples(Y/N)': '1', 'Fast food (Y/N)': '0.0', 'Reg.Exercise(Y/N)': '1', 'BP _Systolic (mmHg)': 105.483401180277, 'BP _Diastolic (mmHg)': 76.09637898063808, 'Follicle No. (L)': 2, 'Follicle No. (R)': 4, 'Avg. F size (L) (mm)': 10.0, 'Avg. F s

In [28]:
structured_result = predictor.predict_structured(patient_data)

print("===== COMPLETE STRUCTURED RESULT =====")
print(structured_result)

===== COMPLETE STRUCTURED RESULT =====
{'label': 'Non-PCOS', 'prob_pcos': 0.0004, 'threshold': 0.5, 'input_completeness': 1.0, 'missing_fields': [], 'warnings': []}


In [29]:
print("Actual PCOS value:", df.iloc[0]["PCOS (Y/N)"])

Actual PCOS value: 0


In [30]:
print("Actual PCOS value:", df.iloc[0]["PCOS (Y/N)"])

Actual PCOS value: 0
